In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rc('font', family='NanumGothic')  # Colab용

plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False


# 결측치 시각화를 위한 라이브러리
import missingno

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 학습 모델 저장을 위한 라이브러리
import pickle

# 프로젝트 셋팅

In [3]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = '/content/drive/MyDrive/파이널 프로젝트/E_NotE/model/best_model_E_NotE.dat'
# 교차검증 횟수
cv_count = 5
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

# 데이터 준비

In [20]:
# 데이터를 읽어온다.
train_df = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트/E_NotE/(E_NotE)_train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/파이널 프로젝트/E_NotE/(E_NotE)_test.csv')

display(train_df)
display(test_df)

,Group,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,Not_E,196,1,1,26,88693,1,1회 이상,11097,7,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,E,13475,1,1,46,16861,1,1회 이상,18638,15,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,Not_E,23988,1,1,28,165221,1,1회 이상,29192,12,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,Not_E,3904,1,1,1,127371,1,1회 이상,18056,8,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,E,1190,1,0,-2,155,0,1회 이상,787,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,10755,1,0,3,0,0,1회 이상,0,0,...,8,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2399996,Not_E,27636,1,1,38,99849,1,1회 이상,60373,7,...,0,23742,17,6,9705,01.100만원+,1회 이상,1회 이상,1회 이상,1회 이상
2399997,Not_E,23187,1,1,33,41073,1,1회 이상,32036,13,...,1,4125,24,6,5346,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상
2399998,E,0,0,0,-2,0,0,1회 이상,0,0,...,12,507,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상


,이용금액대,할인건수_R3M,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M,정상청구원금_B5M,이용금액_R3M_신용체크,이용개월수_신용_R12M,연속유실적개월수_기본_24M_카드,...,_2순위쇼핑업종_이용금액,이용카드수_신용체크,_3순위쇼핑업종_이용금액,정상입금원금_B0M,_3순위업종_이용금액,청구서발송여부_B0,이용후경과월_신용,_2순위업종_이용금액,이용카드수_신용,_1순위카드이용건수
0,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,3919,21458,10,5,...,854,2,643,3680,1880,1,0,2713,2,51
1,02.50만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,5723,18681,9,8,...,1154,2,1033,8726,1278,1,0,1321,1,40
2,01.100만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,11267,40758,12,24,...,1178,2,1094,11297,2680,1,1,7271,2,154
3,04.10만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,5255,9,5,...,804,1,634,1375,605,1,0,2043,1,105
4,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1347,16148,12,20,...,774,3,760,3951,1054,1,0,1364,2,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,0,0,0,...,0,0,0,0,0,0,12,0,0,-2
599996,05.10만원-,1회 이상,10회 이상,1회 이상,1회 이상,1회 이상,992,3110,10,8,...,0,1,0,205,0,1,0,0,1,4
599997,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,0,0,0,0,...,0,0,0,0,0,0,12,0,0,6
599998,01.100만원+,1회 이상,40회 이상,1회 이상,1회 이상,1회 이상,27335,173263,12,24,...,2208,6,1965,0,4612,1,0,16296,4,185


In [21]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,Group,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,Not_E,196,1,1,26,88693,1,1회 이상,11097,7,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,E,13475,1,1,46,16861,1,1회 이상,18638,15,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,Not_E,23988,1,1,28,165221,1,1회 이상,29192,12,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,Not_E,3904,1,1,1,127371,1,1회 이상,18056,8,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,E,1190,1,0,-2,155,0,1회 이상,787,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,NaN,0,0,0,-2,0,0,1회 이상,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999996,NaN,3110,1,1,4,2237,1,1회 이상,2331,5,...,0,992,8,6,205,05.10만원-,10회 이상,1회 이상,1회 이상,1회 이상
2999997,NaN,0,0,0,6,0,0,1회 이상,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999998,NaN,173263,6,4,185,108420,1,1회 이상,89973,66,...,0,27335,24,6,0,01.100만원+,40회 이상,1회 이상,1회 이상,1회 이상


In [22]:
# 결과 데이터는 제거한다.
all_df.drop('Group', axis=1, inplace=True)
all_df

,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,_2순위쇼핑업종_이용금액,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,196,1,1,26,88693,1,1회 이상,11097,7,0,...,0,14958,13,6,6335,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
1,13475,1,1,46,16861,1,1회 이상,18638,15,435,...,0,3367,12,6,5198,03.30만원+,1회 이상,1회 이상,1회 이상,1회 이상
2,23988,1,1,28,165221,1,1회 이상,29192,12,1038,...,0,23963,8,5,12564,01.100만원+,30회 이상,10회 이상,1회 이상,1회 이상
3,3904,1,1,1,127371,1,1회 이상,18056,8,487,...,0,19614,5,6,7639,01.100만원+,1회 이상,1회 이상,10회 이상,10회 이상
4,1190,1,0,-2,155,0,1회 이상,787,0,0,...,6,0,0,1,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,0,0,0,-2,0,0,1회 이상,0,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999996,3110,1,1,4,2237,1,1회 이상,2331,5,0,...,0,992,8,6,205,05.10만원-,10회 이상,1회 이상,1회 이상,1회 이상
2999997,0,0,0,6,0,0,1회 이상,0,0,0,...,12,0,0,0,0,09.미사용,1회 이상,1회 이상,1회 이상,1회 이상
2999998,173263,6,4,185,108420,1,1회 이상,89973,66,2208,...,0,27335,24,6,0,01.100만원+,40회 이상,1회 이상,1회 이상,1회 이상


In [27]:
# LabelEncoder 객체 생성
# ============================
이용금액대Encoder = LabelEncoder()
할인건수_R3MEncoder = LabelEncoder()
방문횟수_앱_R6MEncoder = LabelEncoder()
방문일수_PC_R6MEncoder = LabelEncoder()
인입횟수_ARS_R6MEncoder = LabelEncoder()
이용메뉴건수_ARS_R6MEncoder = LabelEncoder()

# ============================
# all_df 기준으로 fit
# ============================
이용금액대Encoder.fit(all_df['이용금액대'].astype(str))
할인건수_R3MEncoder.fit(all_df['할인건수_R3M'].astype(str))
방문횟수_앱_R6MEncoder.fit(all_df['방문횟수_앱_R6M'].astype(str))
방문일수_PC_R6MEncoder.fit(all_df['방문일수_PC_R6M'].astype(str))
인입횟수_ARS_R6MEncoder.fit(all_df['인입횟수_ARS_R6M'].astype(str))
이용메뉴건수_ARS_R6MEncoder.fit(all_df['이용메뉴건수_ARS_R6M'].astype(str))


LabelEncoder()

In [28]:
# all_df 변환
# ============================
all_df['이용금액대'] = 이용금액대Encoder.transform(all_df['이용금액대'].astype(str))
all_df['할인건수_R3M'] = 할인건수_R3MEncoder.transform(all_df['할인건수_R3M'].astype(str))
all_df['방문횟수_앱_R6M'] = 방문횟수_앱_R6MEncoder.transform(all_df['방문횟수_앱_R6M'].astype(str))
all_df['방문일수_PC_R6M'] = 방문일수_PC_R6MEncoder.transform(all_df['방문일수_PC_R6M'].astype(str))
all_df['인입횟수_ARS_R6M'] = 인입횟수_ARS_R6MEncoder.transform(all_df['인입횟수_ARS_R6M'].astype(str))
all_df['이용메뉴건수_ARS_R6M'] = 이용메뉴건수_ARS_R6MEncoder.transform(all_df['이용메뉴건수_ARS_R6M'].astype(str))

# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

StandardScaler()

In [29]:
# 학습용 데이터(train_df) 변환
# ============================
train_df['이용금액대'] = 이용금액대Encoder.transform(train_df['이용금액대'].astype(str))
train_df['할인건수_R3M'] = 할인건수_R3MEncoder.transform(train_df['할인건수_R3M'].astype(str))
train_df['방문횟수_앱_R6M'] = 방문횟수_앱_R6MEncoder.transform(train_df['방문횟수_앱_R6M'].astype(str))
train_df['방문일수_PC_R6M'] = 방문일수_PC_R6MEncoder.transform(train_df['방문일수_PC_R6M'].astype(str))
train_df['인입횟수_ARS_R6M'] = 인입횟수_ARS_R6MEncoder.transform(train_df['인입횟수_ARS_R6M'].astype(str))
train_df['이용메뉴건수_ARS_R6M'] = 이용메뉴건수_ARS_R6MEncoder.transform(train_df['이용메뉴건수_ARS_R6M'].astype(str))
train_df

,Group,이용금액_R3M_신용체크,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,청구금액_R6M,청구서발송여부_B0,할인건수_R3M,이용금액_오프라인_R6M,이용건수_오프라인_B0M,...,이용후경과월_신용,정상청구원금_B5M,연속유실적개월수_기본_24M_카드,이용개월수_전체_R6M,정상입금원금_B0M,이용금액대,방문횟수_앱_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,Not_E,196,1,1,26,88693,1,1,11097,7,...,0,14958,13,6,6335,0,1,1,0,0
1,E,13475,1,1,46,16861,1,1,18638,15,...,0,3367,12,6,5198,2,1,1,1,1
2,Not_E,23988,1,1,28,165221,1,1,29192,12,...,0,23963,8,5,12564,0,3,0,1,1
3,Not_E,3904,1,1,1,127371,1,1,18056,8,...,0,19614,5,6,7639,0,1,1,0,0
4,E,1190,1,0,-2,155,0,1,787,0,...,6,0,0,1,0,5,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,E,10755,1,0,3,0,0,1,0,0,...,8,0,0,0,0,5,1,1,1,1
2399996,Not_E,27636,1,1,38,99849,1,1,60373,7,...,0,23742,17,6,9705,0,1,1,1,1
2399997,Not_E,23187,1,1,33,41073,1,1,32036,13,...,1,4125,24,6,5346,1,1,1,1,1
2399998,E,0,0,0,-2,0,0,1,0,0,...,12,507,0,0,0,5,1,1,1,1


In [30]:
# 입력과 결과로 나눈다.
X = train_df.drop('Group', axis=1)
y = train_df['Group']

In [31]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.71919385, -0.33371433, -0.19290226, ...,  0.0641234 ,
        -5.69855686, -4.44973122],
       [-0.14848993, -0.33371433, -0.19290226, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [ 0.30333704, -0.33371433, -0.19290226, ..., -3.71384184,
         0.17548303,  0.08322662],
       ...,
       [ 0.26891172, -0.33371433, -0.19290226, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [-0.72761753, -1.33380024, -1.30411597, ...,  0.0641234 ,
         0.17548303,  0.08322662],
       [ 0.19481777,  0.66637157,  0.91831146, ...,  0.0641234 ,
         0.17548303,  0.08322662]])

In [32]:
train_X = X2
train_y = y

In [40]:
train_y = train_y.map({'E': 1, 'Not_E': 0})

# 기본 모델 사용하기

In [34]:
# LGBM
lgbm_basic_model = LGBMClassifier(verbose=-1)
# 교차 검증을 수행한다
r1 = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("LGBM Basic_f1")

In [35]:
# LGBM
lgbm_basic_model = LGBMClassifier(verbose=-1)
# 교차 검증을 수행한다
r1 = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1_micro', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("LGBM Basic_f1_micro")

In [37]:
# LGBM
lgbm_basic_model = LGBMClassifier(verbose=-1)
# 교차 검증을 수행한다
r1 = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1_macro', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("LGBM Basic_f1_micro")

In [49]:
# XGBoost
xgboost_basic_model1 = XGBClassifier(
    tree_method='gpu_hist',    # GPU 히스토그램 방식
    predictor='gpu_predictor', # GPU 예측 사용
    gpu_id=0,                  # GPU ID (보통 0)
    verbosity=0                # 출력 레벨 (0: silent)
)

# 교차 검증 수행
r1 = cross_val_score(xgboost_basic_model, train_X, train_y, scoring='f1', cv=kfold)

# 평가 결과 저장
f1_score_list.append(r1.mean())

# 모델 이름 저장
model_name_list.append("XGBoost Basic_f1")

In [46]:
# XGBoost
xgboost_basic_model2 = XGBClassifier(
    tree_method='gpu_hist',    # GPU 히스토그램 방식
    predictor='gpu_predictor', # GPU 예측 사용
    gpu_id=0,                  # GPU ID (보통 0)
    verbosity=0                # 출력 레벨 (0: silent)
)

# 교차 검증 수행
r1 = cross_val_score(xgboost_basic_model, train_X, train_y, scoring='f1_macro', cv=kfold)

# 평가 결과 저장
f1_score_list.append(r1.mean())

# 모델 이름 저장
model_name_list.append("XGBoost Basic_f1_macro")

In [47]:
# XGBoost
xgboost_basic_model3= XGBClassifier(
    tree_method='gpu_hist',    # GPU 히스토그램 방식
    predictor='gpu_predictor', # GPU 예측 사용
    gpu_id=0,                  # GPU ID (보통 0)
    verbosity=0                # 출력 레벨 (0: silent)
)

# 교차 검증 수행
r1 = cross_val_score(xgboost_basic_model, train_X, train_y, scoring='f1_micro', cv=kfold)

# 평가 결과 저장
f1_score_list.append(r1.mean())

# 모델 이름 저장
model_name_list.append("XGBoost Basic_micro")

In [48]:
d1 = {
    'f1 score' : f1_score_list
}
result_df = pd.DataFrame(d1, index=model_name_list)
result_df.sort_values(by='f1 score', ascending=False, inplace=True)
result_df

,f1 score
XGBoost Basic_f1,0.943085
XGBoost Basic,0.943074
XGBoost Basic (GPU),0.907406
XGBoost Basic_micro,0.907406
LGBM Basic_f1_micro,0.904821
XGBoost Basic (GPU),0.847465
XGBoost Basic_f1_macro,0.847465
LGBM Basic_f1_micro,0.842589
LGBM Basic,NaN
LGBM Basic_f1,NaN


In [50]:
final_model=xgboost_basic_model1.fit(train_X, train_y)

In [53]:
import pickle

# 저장할 경로
best_model_path = '/content/drive/MyDrive/파이널 프로젝트/E_NotE/model/best_model_E_NotE_1.dat'

with open(best_model_path, 'wb') as fp:
    pickle.dump(final_model, fp)
    pickle.dump(scalerX, fp)  # 스케일러 저장
    # 필요하다면 인코더도 추가로 저장
    pickle.dump(le, fp)       # LabelEncoder 저장
    # 추가적으로 필요한 LabelEncoder들
    pickle.dump(이용금액대Encoder, fp)
    pickle.dump(할인건수_R3MEncoder, fp)
    pickle.dump(방문횟수_앱_R6MEncoder, fp)
    pickle.dump(방문일수_PC_R6MEncoder, fp)
    pickle.dump(인입횟수_ARS_R6MEncoder, fp)
    pickle.dump(이용메뉴건수_ARS_R6MEncoder, fp)

print('✅ 모델과 인코더 저장 완료!')

✅ 모델과 인코더 저장 완료!
